In [1]:
# pygame  -> lets us make 2D games (drawing, input, sound, etc.)
# random  -> used to pick random numbers (e.g. random pipe heights)
# math    -> used for sin() to create smooth wave-like animations (wings, bobbing)
import pygame
import random  
import math
import os         

pygame.init()  # Starts up all of pygame's internal systems. Must be called first.
pygame.mixer.init()  # Starts up the sound system. Must be called before loading/playing any sound.

# ---------------- Setup Game window ----------------
screen_width = 800
screen_height = 600
# This creates the actual window the game will be drawn into.
# Everything we draw (screen.blit / pygame.draw...) shows up on this "screen" surface.
screen = pygame.display.set_mode((screen_width, screen_height))
pygame.display.set_caption("Ninja Escape")  # Text shown in the window's title bar
clock = pygame.time.Clock()  # Used later to lock the game to 60 frames per second

# ---------------- Colors ----------------
white = (255, 255, 255)
black = (0, 0, 0)
red = (220, 50, 50)
gray = (100, 100, 100)
dark_gray = (60, 60, 60)

SKY_TOP = (100, 180, 235)
SKY_BOTTOM = (200, 235, 250)
SUN_COLOR = (255, 240, 180)

GROUND_TOP = (222, 196, 107)
GROUND_BOTTOM = (180, 150, 70)
GRASS_TOP = (110, 200, 80)
GRASS_DARK = (80, 170, 60)

PIPE_MAIN = (70, 190, 90)
PIPE_LIGHT = (120, 225, 130)
PIPE_DARK = (40, 140, 60)
PIPE_EDGE = (25, 90, 40)
PIPE_HILITE = (190, 255, 200)   # bright shine strip -> makes the pipe look round/glossy
PIPE_RIVET = (30, 80, 45)       # dark bolt/rivet color

BIRD_BEAK = (255, 130, 40)      # kept in case you want a drawn fallback later
# Colors used for the little feather/impact particles, matched to a scarlet
# macaw's plumage (red, blue, yellow) instead of the old drawn bird's colors.
FEATHER_COLORS = [(210, 40, 40), (40, 90, 200), (240, 200, 40)]
BIRD_BEAK_DARK = (220, 90, 20)

# ---------------- Ground / floor ----------------
ground_height = 90
# play_bottom is the y-coordinate where the "floor" starts.
# The bird is allowed to fly anywhere ABOVE this line.
play_bottom = screen_height - ground_height 

# ---------------- Bird settings ----------------
bird_x = 120        # The bird never moves left/right — only up/down. This is its fixed x position.
bird_radius = 22     # Half the bird's width/height, used for drawing and collision.

# ---------------- Pipe settings ----------------
pipe_width = 90
pipe_gap = 165        # Vertical gap the bird must fly through
pipe_speed = 3         # How many pixels pipes move left, every frame
pipe_cap_h = 26        # Height of the decorative "cap" at the end of each pipe
pipe_cap_over = 8      # How far the cap sticks out sideways past the pipe body

# ---------------- Physics ----------------
# In pygame, y increases DOWNWARD (0 = top of screen).
gravity = 0.5         # Added to bird_y_change every frame -> constantly pulls the bird down
jump_strength = -6    # Negative = upward. Set as the bird's speed the instant you flap.

# ---------------- Fonts ----------------
font_score = pygame.font.SysFont("arial", 46, bold=True)
font_big = pygame.font.SysFont("arial", 54, bold=True)
font_med = pygame.font.SysFont("arial", 26, bold=True)
font_small = pygame.font.SysFont("arial", 20)


def draw_text_outline(surface, text, font, x, y, fg, outline, center=True, width=3):
    """Draws text with a colored outline around it, so it stays readable
    over any background (sky, pipes, ground, etc).
    Trick: draw the text several times in the outline color, slightly
    offset in every direction, THEN draw the real text on top in the center."""
    base = font.render(text, True, fg)
    rect = base.get_rect()
    if center:
        rect.center = (x, y)
    else:
        rect.topleft = (x, y)
    for dx in range(-width, width + 1, width):
        for dy in range(-width, width + 1, width):
            if dx == 0 and dy == 0:
                continue
            oshadow = font.render(text, True, outline)
            orect = oshadow.get_rect(center=(rect.centerx + dx, rect.centery + dy)) if center \
                else oshadow.get_rect(topleft=(rect.x + dx, rect.y + dy))
            surface.blit(oshadow, orect)
    surface.blit(base, rect)


def vertical_gradient(surface, rect, top_color, bottom_color):
    """Fills a rectangle with a smooth color blend from top_color to bottom_color.
    This is what makes the sky, ground, and pipes look 3D instead of flat.
    How: draw ONE horizontal line per pixel-row, each a slightly different
    mix ("interpolation") between the top and bottom color."""
    x, y, w, h = rect
    if h <= 0:
        return
    for i in range(h):
        t = i / max(h - 1, 1)
        color = tuple(int(top_color[c] + (bottom_color[c] - top_color[c]) * t) for c in range(3))
        pygame.draw.line(surface, color, (x, y + i), (x + w, y + i))


def horizontal_gradient(surface, rect, color_stops):
    """Fills a rectangle left-to-right using several color 'stops' instead of
    just two colors. This is what makes the pipe look like a round tube:
    dark at the edges, a bright shine near the left-center, fading back to
    the base color, then dark again at the right edge — like real light
    hitting a cylinder.
    color_stops: list of (position 0.0-1.0, (r,g,b)), sorted left to right."""
    x, y, w, h = rect
    if w <= 0 or h <= 0:
        return
    for i in range(w):
        t = i / max(w - 1, 1)
        # find which two stops 't' falls between, then blend those two colors
        for j in range(len(color_stops) - 1):
            t0, c0 = color_stops[j]
            t1, c1 = color_stops[j + 1]
            if t0 <= t <= t1:
                local_t = 0 if t1 == t0 else (t - t0) / (t1 - t0)
                color = tuple(int(c0[c] + (c1[c] - c0[c]) * local_t) for c in range(3))
                break
        else:
            color = color_stops[-1][1]
        pygame.draw.line(surface, color, (x + i, y), (x + i, y + h))


# ---------------- Background layer (pre-rendered sky) ----------------
# Apni image file ka naam/path yahan daalo (same folder me rakho image)
sky_surface = pygame.image.load("back.jpg").convert()
sky_surface = pygame.transform.scale(sky_surface, (screen_width, screen_height))


# ---------------- Bird sprite (single image) ----------------
BIRD_IMAGE_FILE = "ninja.webp"   # your uploaded image, kept in the same folder as this script
BIRD_SPRITE_SIZE = (100, 60)              # (width, height) in pixels the bird is drawn at in-game

# Folder this script lives in, so the image/sound files load correctly no
# matter where you run the game from.
try:
    _script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _script_dir = os.getcwd()


def load_bird_image():
    """Loads BIRD_IMAGE_FILE and scales it to BIRD_SPRITE_SIZE."""
    path = os.path.join(_script_dir, BIRD_IMAGE_FILE)
    try:
        # convert_alpha() keeps the image's transparent background transparent
        img = pygame.image.load(path).convert_alpha()
        return pygame.transform.smoothscale(img, BIRD_SPRITE_SIZE)
    except (pygame.error, FileNotFoundError) as e:
        raise SystemExit(
            f"Could not load bird image '{BIRD_IMAGE_FILE}': {e}\n"
            f"Make sure it's sitting right next to this script."
        )


bird_image = load_bird_image()


# ==================== SOUND SETUP (NEW) ====================
# Filenames for the 3 sounds you asked for. Drop matching files in the
# same folder as this script. .wav or .ogg work best for pygame.
THEME_SOUND_FILE = "music.mp3"          # background music, loops the whole time you're playing
COIN_SOUND_FILE = "coin.mp3"            # plays every time the bird passes/collects a pipe "coin"
GAMEOVER_SOUND_FILE = "Game over.mp3"    # plays once when you crash


def load_sound(filename, volume=1.0):
    """Loads a short sound effect (Coin, Game Over). Returns None (and just
    prints a warning) if the file is missing, so the game won't crash."""
    path = os.path.join(_script_dir, filename)
    try:
        s = pygame.mixer.Sound(path)
        s.set_volume(volume)
        return s
    except (pygame.error, FileNotFoundError) as e:
        print(f"Warning: could not load sound '{filename}': {e}")
        return None


def play_sound(sound):
    if sound:
        sound.play()


# Coin sound and Game Over sound: short one-shot effects -> use pygame.mixer.Sound
coin_sound = load_sound(COIN_SOUND_FILE, 0.6)
gameover_sound = load_sound(GAMEOVER_SOUND_FILE, 0.7)

# Theme song: long looping background track -> use pygame.mixer.music instead
# of pygame.mixer.Sound (music is streamed from disk, better for long files,
# and only one music track can play at a time, which is exactly what we want).
try:
    pygame.mixer.music.load(os.path.join(_script_dir, THEME_SOUND_FILE))
    pygame.mixer.music.set_volume(0.35)
    theme_loaded = True
except pygame.error as e:
    print(f"Warning: could not load theme song '{THEME_SOUND_FILE}': {e}")
    theme_loaded = False


def start_theme():
    """Starts the theme song looping forever (loops=-1). Called every time
    a new run begins (start screen -> playing)."""
    if theme_loaded:
        pygame.mixer.music.play(loops=-1)


def stop_theme():
    """Stops the theme song. Called the instant the bird crashes."""
    if theme_loaded:
        pygame.mixer.music.stop()
# ==================== END SOUND SETUP ====================


class Particle:
    """Small feather / dust particle used for flap and impact effects."""
    def __init__(self, x, y, color, vx=None, vy=None, life=30, gravity_on=True, radius=3):
        self.x = x
        self.y = y
        self.vx = vx if vx is not None else random.uniform(-2, 2)
        self.vy = vy if vy is not None else random.uniform(-3, -1)
        self.life = life
        self.max_life = life
        self.color = color
        self.gravity_on = gravity_on
        self.radius = radius

    def update(self):
        self.x += self.vx
        self.y += self.vy
        if self.gravity_on:
            self.vy += 0.15
        self.life -= 1

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = max(0, min(255, int(255 * (self.life / self.max_life))))
        r = max(1, int(self.radius * (self.life / self.max_life)))
        s = pygame.Surface((r * 2 + 2, r * 2 + 2), pygame.SRCALPHA)
        pygame.draw.circle(s, (*self.color, alpha), (r + 1, r + 1), r)
        surf.blit(s, (self.x - r, self.y - r))

    @property
    def alive(self):
        return self.life > 0


def draw_ground(surf, scroll_x):
    ground_rect = (0, play_bottom, screen_width, ground_height)
    vertical_gradient(surf, ground_rect, GROUND_TOP, GROUND_BOTTOM)
    # grass strip
    pygame.draw.rect(surf, GRASS_TOP, (0, play_bottom, screen_width, 14))
    pygame.draw.rect(surf, GRASS_DARK, (0, play_bottom, screen_width, 4))
    # scrolling grass tufts
    tuft_w = 20
    offset = int(scroll_x) % tuft_w
    x = -offset
    while x < screen_width:
        pygame.draw.polygon(
            surf, GRASS_DARK,
            [(x, play_bottom + 6), (x + tuft_w // 2, play_bottom - 4), (x + tuft_w, play_bottom + 6)]
        )
        x += tuft_w
    # subtle plank lines on dirt
    line_w = 40
    offset2 = int(scroll_x * 1.5) % line_w
    x = -offset2
    while x < screen_width:
        pygame.draw.line(surf, (170, 140, 60), (x, play_bottom + 20), (x, screen_height), 3)
        x += line_w


def draw_pipe_body(surf, x, y, w, h):
    """Draws one straight pipe segment (just the tube part, no cap).
    Uses horizontal_gradient to fake roundness, then adds a couple of
    horizontal 'seam' lines like a real pipe is built from joined sections."""
    if h <= 0:
        return
    stops = [
        (0.00, PIPE_EDGE),
        (0.10, PIPE_DARK),
        (0.24, PIPE_HILITE),   # the shiny highlight strip
        (0.36, PIPE_LIGHT),
        (0.55, PIPE_MAIN),
        (0.80, PIPE_DARK),
        (1.00, PIPE_EDGE),
    ]
    horizontal_gradient(surf, (x, y, w, h), stops)

    # horizontal seam lines every ~34px, like joints between pipe sections
    seam_gap = 34
    yy = y + seam_gap
    while yy < y + h - 4:
        pygame.draw.line(surf, PIPE_EDGE, (x, yy), (x + w, yy), 1)
        pygame.draw.line(surf, PIPE_HILITE, (x, yy + 1), (x + w, yy + 1), 1)
        yy += seam_gap

    pygame.draw.rect(surf, PIPE_EDGE, (x, y, w, h), 2)  # crisp outline


def draw_pipe_cap(surf, x, y, w, h):
    """Draws the wider rim/collar at the open end of a pipe, with the
    same cylindrical shading plus a row of rivets for a metal-pipe feel."""
    stops = [
        (0.00, PIPE_EDGE),
        (0.08, PIPE_DARK),
        (0.22, PIPE_HILITE),
        (0.38, PIPE_LIGHT),
        (0.60, PIPE_MAIN),
        (0.85, PIPE_DARK),
        (1.00, PIPE_EDGE),
    ]
    rect = pygame.Rect(x, y, w, h)
    horizontal_gradient(surf, rect, stops)
    pygame.draw.rect(surf, PIPE_EDGE, rect, 2, border_radius=4)

    # rivets: small bolt-like circles with a tiny highlight dot for shine
    rivet_y = y + h // 2
    for rx in range(x + 10, x + w - 8, 16):
        pygame.draw.circle(surf, PIPE_RIVET, (rx, rivet_y), 3)
        pygame.draw.circle(surf, PIPE_HILITE, (rx - 1, rivet_y - 1), 1)


def draw_pipe_pair(surf, pipe_x, gap_top):
    gap_bottom = gap_top + pipe_gap

    # ---- Top pipe: body first, cap drawn on top so it overlaps cleanly ----
    draw_pipe_body(surf, pipe_x, 0, pipe_width, gap_top)
    draw_pipe_cap(surf, pipe_x - pipe_cap_over, gap_top - pipe_cap_h,
                  pipe_width + pipe_cap_over * 2, pipe_cap_h)

    # ---- Bottom pipe: cap first, then body below it ----
    draw_pipe_cap(surf, pipe_x - pipe_cap_over, gap_bottom,
                  pipe_width + pipe_cap_over * 2, pipe_cap_h)
    draw_pipe_body(surf, pipe_x, gap_bottom + pipe_cap_h,
                   pipe_width, screen_height - gap_bottom - pipe_cap_h)


def draw_bird(surf, x, y, angle, wing_phase):
    """Draws your bird image, with two small tricks to make a single static
    photo feel alive:
    1) A gentle "squash-stretch" pulse (image gets slightly taller/thinner
       then shorter/wider on a loop) using sin(wing_phase) — this fakes a
       wingbeat without needing multiple images.
    2) Rotation based on velocity, so it tilts up when climbing and dips
       down when falling, same as before."""
    # pulse_t swings smoothly between -1 and 1 forever
    pulse_t = math.sin(wing_phase)
    stretch = 1 + 0.08 * pulse_t     # height grows/shrinks by up to 8%
    squeeze = 1 - 0.05 * pulse_t     # width does the opposite, subtly

    w, h = BIRD_SPRITE_SIZE
    pulsed = pygame.transform.smoothscale(bird_image, (int(w * squeeze), int(h * stretch)))

    rotated = pygame.transform.rotate(pulsed, angle)
    rect = rotated.get_rect(center=(x, y))
    surf.blit(rotated, rect)



def show_score(surf, score, best):
    draw_text_outline(surf, str(score), font_score, screen_width // 2, 60, white, black, width=3)
    small = font_small.render(f"Best: {best}", True, white)
    small_bg = font_small.render(f"Best: {best}", True, black)
    surf.blit(small_bg, (12, 12))
    surf.blit(small, (10, 10))


def flash_score_popup(particles_list, x, y):
    for _ in range(10):
        particles_list.append(
            Particle(x, y, (255, 255, 255), vx=random.uniform(-1.5, 1.5),
                     vy=random.uniform(-2, -0.5), life=20, gravity_on=False, radius=2)
        )


def reset_state():
    """Creates one dictionary holding everything that changes during a run:
    bird position/speed, the list of pipes on screen, score, animation
    timers, particle effects, and screen-shake. Calling this = fresh game."""
    return {
        "bird_y": screen_height // 2,       # bird's current vertical position
        "bird_y_change": 0,                  # bird's current vertical speed (velocity)
        # Each pipe carries its OWN "scored" flag. This avoids a subtle bug where
        # tracking scored pipes in a separate set by id(p) breaks once old pipe
        # dicts get garbage-collected and their id() gets reused by new pipes.
        "pipes": [{"x": screen_width + 100, "gap_top": random.randint(120, play_bottom - 300), "scored": False}],
        "score": 0,
        "wing_phase": 0.0,                   # feeds into sin() for the wing-flap animation
        "scroll": 0.0,                       # how far the ground texture has scrolled
        "particles": [],                     # list of active Particle objects (feathers/dust)
        "shake": 0,                          # frames of screen-shake left after a crash
    }


def spawn_flap_particles(particles_list, x, y):
    for _ in range(6):
        particles_list.append(
            Particle(x - 10, y, random.choice(FEATHER_COLORS), vx=random.uniform(-2, 0.5),
                     vy=random.uniform(0.5, 2.5), life=22, gravity_on=True, radius=3)
        )


def spawn_impact_particles(particles_list, x, y):
    for _ in range(24):
        particles_list.append(
            Particle(x, y, random.choice(FEATHER_COLORS + [white]),
                     vx=random.uniform(-4, 4), vy=random.uniform(-5, 1),
                     life=40, gravity_on=True, radius=4)
        )


def draw_start_screen(surf, bird_bob):
    surf.blit(sky_surface, (0, 0))
    draw_ground(surf, 0)
    draw_bird(surf, screen_width // 2, screen_height // 2 - 40 + bird_bob, 0, bird_bob * 0.4)
    draw_text_outline(surf, "FLAPPY BIRD", font_big, screen_width // 2, 140, (255, 220, 60), black, width=3)
    draw_text_outline(surf, "Press SPACE to Start", font_med, screen_width // 2, screen_height // 2 + 90, white, black, width=2)
    draw_text_outline(surf, "SPACE / Click to flap", font_small, screen_width // 2, screen_height // 2 + 130, white, black, width=1)


def main_game():
    """This is the heart of the game: the MAIN GAME LOOP.
    A game loop is just: 1) read input -> 2) update the world -> 3) draw
    everything -> 4) repeat, 60 times per second. We track which "screen"
    we're on using state_name: "start" -> "playing" -> "gameover" -> repeat."""
    best_score = 0
    state_name = "start"
    state = reset_state()
    start_bob_t = 0.0

    running = True
    while running:
        # ---- 1) INPUT: check what the player did this frame ----
        dt_events = pygame.event.get()
        flap = False  # will become True if the player pressed space / clicked this frame
        for event in dt_events:
            if event.type == pygame.QUIT:          # player clicked the window's X button
                pygame.quit()
                return
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:
                    flap = True
                if event.key == pygame.K_r and state_name == "gameover":
                    state = reset_state()
                    state_name = "playing"
                    start_theme()  # SOUND: theme song starts again on restart
            if event.type == pygame.MOUSEBUTTONDOWN:
                flap = True

        if state_name == "start":
            start_bob_t += 0.08
            bob = math.sin(start_bob_t) * 8
            if flap:
                state = reset_state()
                state_name = "playing"
                start_theme()  # SOUND: theme song starts the moment gameplay begins
            draw_start_screen(screen, bob)

        elif state_name == "playing":
            # ---- 2) UPDATE: move everything forward by one frame ----

            # Flapping just SETS the vertical speed instantly upward.
            # Gravity below will immediately start pulling it back down again,
            # which is what gives Flappy Bird its floaty "jump-then-fall" feel.
            if flap:
                state["bird_y_change"] = jump_strength
                spawn_flap_particles(state["particles"], bird_x, state["bird_y"])

            state["bird_y_change"] += gravity      # gravity constantly speeds up the fall
            state["bird_y"] += state["bird_y_change"]  # apply that speed to the position
            state["wing_phase"] += 0.35            # advances the wing-flap animation
            state["scroll"] += pipe_speed          # makes the ground texture scroll

            # move pipes, spawn new ones
            for p in state["pipes"]:
                p["x"] -= pipe_speed
            if state["pipes"][-1]["x"] < screen_width - 300:
                state["pipes"].append({
                    "x": screen_width + 40,
                    "gap_top": random.randint(90, play_bottom - pipe_gap - 90),
                    "scored": False,
                })
            if state["pipes"][0]["x"] < -pipe_width - 20:
                state["pipes"].pop(0)

            # scoring — check each pipe's OWN "scored" flag, not a shared id() set
            # This is treated as "collecting the coin" in that pipe's gap.
            for p in state["pipes"]:
                if p["x"] + pipe_width < bird_x and not p["scored"]:
                    p["scored"] = True
                    state["score"] += 1
                    flash_score_popup(state["particles"], bird_x + 30, state["bird_y"])
                    play_sound(coin_sound)  # SOUND: coin sound plays every time a pipe is passed

            # ---- Collision detection ----
            # pygame.Rect is an invisible rectangle we use just for math.
            # colliderect() checks if two rectangles overlap -> that's our crash test.
            bird_rect = pygame.Rect(
                bird_x - bird_radius, state["bird_y"] - bird_radius,
                bird_radius * 2, bird_radius * 2
            )
            hit = False
            for p in state["pipes"]:
                top_rect = pygame.Rect(p["x"], 0, pipe_width, p["gap_top"])
                bottom_rect = pygame.Rect(
                    p["x"], p["gap_top"] + pipe_gap, pipe_width,
                    screen_height - p["gap_top"] - pipe_gap
                )
                if bird_rect.colliderect(top_rect) or bird_rect.colliderect(bottom_rect):
                    hit = True
            if state["bird_y"] - bird_radius < 0 or state["bird_y"] + bird_radius > play_bottom:
                hit = True

            if hit:
                state_name = "gameover"
                best_score = max(best_score, state["score"])
                spawn_impact_particles(state["particles"], bird_x, state["bird_y"])
                state["shake"] = 14
                stop_theme()               # SOUND: theme song stops the instant you crash
                play_sound(gameover_sound)  # SOUND: game over sound plays once

            # update particles
            state["particles"] = [pt for pt in state["particles"] if pt.alive]
            for pt in state["particles"]:
                pt.update()

            # ---- 3) DRAW: paint the current frame, back-to-front ----
            # Order matters! We draw sky first, then pipes, then ground,
            # then particles, then the bird, then the score on top —
            # so nearer things cover farther things, like layers.
            shake_x = random.randint(-2, 2) if state["shake"] > 0 else 0
            shake_y = random.randint(-2, 2) if state["shake"] > 0 else 0
            if state["shake"] > 0:
                state["shake"] -= 1

            screen.blit(sky_surface, (0, 0))
            for p in state["pipes"]:
                draw_pipe_pair(screen, p["x"] + shake_x, p["gap_top"])
            draw_ground(screen, state["scroll"] + shake_x)
            for pt in state["particles"]:
                pt.draw(screen)

            angle = max(-25, min(70, -state["bird_y_change"] * 3))
            draw_bird(screen, bird_x + shake_x, state["bird_y"] + shake_y, angle, state["wing_phase"])

            show_score(screen, state["score"], best_score)

        elif state_name == "gameover":
            state["wing_phase"] += 0.05
            state["particles"] = [pt for pt in state["particles"] if pt.alive]
            for pt in state["particles"]:
                pt.update()

            screen.blit(sky_surface, (0, 0))
            for p in state["pipes"]:
                draw_pipe_pair(screen, p["x"], p["gap_top"])
            draw_ground(screen, state["scroll"])
            for pt in state["particles"]:
                pt.draw(screen)
            draw_bird(screen, bird_x, min(state["bird_y"], play_bottom - bird_radius), 80, 0)

            overlay = pygame.Surface((screen_width, screen_height), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 90))
            screen.blit(overlay, (0, 0))

            panel = pygame.Rect(screen_width // 2 - 180, screen_height // 2 - 110, 360, 220)
            pygame.draw.rect(screen, (245, 235, 210), panel, border_radius=14)
            pygame.draw.rect(screen, black, panel, 3, border_radius=14)

            draw_text_outline(screen, "GAME OVER", font_big, screen_width // 2, panel.y + 45, red, white, width=2)
            draw_text_outline(screen, f"Score: {state['score']}", font_med, screen_width // 2, panel.y + 105, black, white, width=1)
            draw_text_outline(screen, f"Best: {best_score}", font_med, screen_width // 2, panel.y + 140, black, white, width=1)
            draw_text_outline(screen, "Press R or SPACE to Restart", font_small, screen_width // 2, panel.y + 185, dark_gray, white, width=1)

            if flap:
                state = reset_state()
                state_name = "playing"
                start_theme()  # SOUND: theme song starts again when you flap to restart

        # ---- 4) Show it & repeat ----
        pygame.display.update()  # actually pushes everything we drew onto the screen
        clock.tick(60)           # pauses just enough to cap the loop at 60 frames per second

    pygame.quit()


if __name__ == "__main__":
    main_game()

C:\Users\ADMIN\anaconda3\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


False
